<a href="https://colab.research.google.com/github/brunakv/ciencia_dados_II/blob/main/ciencia_dados_II_bruna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark
from pyspark.sql import SparkSession

from pyspark.sql.functions import sum
from pyspark.sql.functions import col
import builtins # Importa o módulo builtins para acessar a função sum nativa do Python

# Inicializando SparkSession
spark = SparkSession.builder \
    .appName("CreditoRuralBrasil") \
    .getOrCreate()

In [2]:
# =====================================================
# ETAPA 1 - SELEÇÃO DO DATASET
# =====================================================

import requests
url = "https://olinda.bcb.gov.br/olinda/servico/SICOR/versao/v2/odata/RegiaoUF?$format=json"

response = requests.get(url)
response.raise_for_status()
dados = response.json()
df_spark = spark.createDataFrame(dados["value"])

In [3]:
# =====================================================
# ETAPA 2 - INGESTÃO E PRÉ-PROCESSAMENTO COM SPARK
# =====================================================

# LIMPEZA E PRÉ-PROCESSAMENTO
df_spark = df_spark.na.fill(0) # Preenche nulos com 0
df_spark = df_spark.dropDuplicates() # Remove linhas duplicadas

# ORDENA PELO ANO DE EMISSÃO (CRESCENTE)
df_spark = df_spark.orderBy("AnoEmissao")

df_spark.show()

+----------+---------+----------+------------------+----------+-------------------+---------------+-----------------+----------+------------------+--------------+--------+--------------+----------+--------+-------------+------------+------+
|AnoEmissao|Atividade|MesEmissao|QtdComercializacao|QtdCusteio|QtdIndustrializacao|QtdInvestimento|VlComercializacao| VlCusteio|VlIndustrializacao|VlInvestimento|cdEstado|cdFonteRecurso|cdPrograma|cdRegiao|cdSubPrograma|  nomeRegiao|nomeUF|
+----------+---------+----------+------------------+----------+-------------------+---------------+-----------------+----------+------------------+--------------+--------+--------------+----------+--------+-------------+------------+------+
|      2013|        1|        02|                 0|         0|                  0|             15|              0.0|       0.0|               0.0|    1584045.57|      24|          0450|      0050|       4|            0|         SUL|    RS|
|      2013|        1|        12|   

In [4]:
# NOME DAS COLUNAS E TIPO DE DADOS
df_spark.printSchema()

# NÚMERO TOTAL DE LINHAS
print(df_spark.count())

root
 |-- AnoEmissao: string (nullable = true)
 |-- Atividade: string (nullable = true)
 |-- MesEmissao: string (nullable = true)
 |-- QtdComercializacao: long (nullable = true)
 |-- QtdCusteio: long (nullable = true)
 |-- QtdIndustrializacao: long (nullable = true)
 |-- QtdInvestimento: long (nullable = true)
 |-- VlComercializacao: double (nullable = false)
 |-- VlCusteio: double (nullable = false)
 |-- VlIndustrializacao: double (nullable = false)
 |-- VlInvestimento: double (nullable = false)
 |-- cdEstado: string (nullable = true)
 |-- cdFonteRecurso: string (nullable = true)
 |-- cdPrograma: string (nullable = true)
 |-- cdRegiao: string (nullable = true)
 |-- cdSubPrograma: string (nullable = true)
 |-- nomeRegiao: string (nullable = true)
 |-- nomeUF: string (nullable = true)

195708


In [5]:
# =====================================================
# ETAPA 3 - ANÁLISE EXPLORATÓRIA COM SPARK SQL
# =====================================================

# CRIA NOVA COLUNA VL_TOTAL: SOMA VlCusteio, VlInvestimento, VlComercializacao, VlIndustrializacao

df_spark = df_spark.withColumn(
    "VL_TOTAL",
    col("VlCusteio") +
    col("VlInvestimento") +
    col("VlComercializacao") +
    col("VlIndustrializacao")
)

# CRIA UMA VIEW TEMPORÁRIA PARA CONSULTAS SPARK SQL
df_spark.createOrReplaceTempView("credito_rural")

# PERGUNTA 1 - ESTADOS COM MAIOR VOLUME DE CRÉDITO RURAL

estados_sql = spark.sql("""
SELECT
    nomeUF,
    SUM(VL_TOTAL) AS TOTAL
FROM credito_rural
GROUP BY nomeUF
ORDER BY TOTAL DESC
""")

estados_sql.show(10)

top10 = estados_sql.limit(10).select("nomeUF").collect()

lista_estados = [linha["nomeUF"] for linha in top10]

texto = (
    "Os estados com maior volume de crédito rural são: "
    + ", ".join(lista_estados[:-1])
    + " e "
    + lista_estados[-1]
    + "."
)

print(texto)

+------+--------------------+
|nomeUF|               TOTAL|
+------+--------------------+
|    PR|4.748818556687203...|
|    RS|4.510496117330104E11|
|    MG|4.218171011536693E11|
|    MT|3.438141419235802...|
|    SP|3.389054098443196E11|
|    GO|  3.0113211262955E11|
|    SC|1.924508716355500...|
|    MS|1.843536811390099...|
|    BA|1.302289101851096...|
|    TO|7.905923768779996E10|
+------+--------------------+
only showing top 10 rows
Os estados com maior volume de crédito rural são: PR, RS, MG, MT, SP, GO, SC, MS, BA e TO.


In [6]:
# PERGUNTA 2 - EVOLUÇÃO ANUAL DO CRÉDITO RURAL

evolucao_anual = spark.sql("""
SELECT
    AnoEmissao,
    SUM(VL_TOTAL) AS VolumeTotal
FROM credito_rural
GROUP BY AnoEmissao
ORDER BY AnoEmissao
""")

evolucao_anual.show()

dados = evolucao_anual.collect()

texto_historico = "A evolução do crédito rural ao longo dos anos foi a seguinte:\n\n"

for linha in dados:
    texto_historico += (
        f"- {linha['AnoEmissao']}: "
        f"R$ {linha['VolumeTotal']:,.2f}\n"
    )

print(texto_historico)

primeiro_ano = dados[0]["AnoEmissao"]
ultimo_ano = dados[-1]["AnoEmissao"]

primeiro_valor = dados[0]["VolumeTotal"]
ultimo_valor = dados[-1]["VolumeTotal"]

variacao = ((ultimo_valor - primeiro_valor) / primeiro_valor) * 100

print(
    f"Entre {primeiro_ano} e {ultimo_ano}, houve variação de "
    f"{variacao:.2f}% no volume de crédito rural."
)

+----------+--------------------+
|AnoEmissao|         VolumeTotal|
+----------+--------------------+
|      2013|1.393859855320402...|
|      2014|1.644308828048694...|
|      2015|1.541460630198897...|
|      2016|1.179041473500599...|
|      2017|1.670198922658704...|
|      2018|1.808912000253905...|
|      2019|1.789626140696293...|
|      2020|  2.0705592642261E11|
|      2021|2.952083585152595E11|
|      2022|3.629806141102119E11|
|      2023|4.065567877569322E11|
|      2024|3.799457858774697E11|
|      2025|3.608863650243191...|
|      2026|2.053402877480511...|
+----------+--------------------+

A evolução do crédito rural ao longo dos anos foi a seguinte:

- 2013: R$ 139,385,985,532.04
- 2014: R$ 164,430,882,804.87
- 2015: R$ 154,146,063,019.89
- 2016: R$ 117,904,147,350.06
- 2017: R$ 167,019,892,265.87
- 2018: R$ 180,891,200,025.39
- 2019: R$ 178,962,614,069.63
- 2020: R$ 207,055,926,422.61
- 2021: R$ 295,208,358,515.26
- 2022: R$ 362,980,614,110.21
- 2023: R$ 406,556,787,7

In [7]:
# PERGUNTA 3 - CRÉDITO POR REGIÃO

volume_regiao = spark.sql("""
SELECT
    nomeRegiao,
    SUM(VL_TOTAL) AS VolumeTotal
FROM credito_rural
GROUP BY nomeRegiao
ORDER BY VolumeTotal DESC
""")

volume_regiao.show()

dados = volume_regiao.collect()

texto = (
    f"A região {dados[0]['nomeRegiao']} apresentou o maior volume de crédito rural. "
    f"Em seguida aparecem as regiões {dados[1]['nomeRegiao']} e "
    f"{dados[2]['nomeRegiao']}."
)

print(texto)

+------------+--------------------+
|  nomeRegiao|         VolumeTotal|
+------------+--------------------+
|         SUL|1.118382339037282E12|
|CENTRO-OESTE|8.333171457660482E11|
|     SUDESTE|8.237615158104294E11|
|    NORDESTE|3.231444349915702E11|
|       NORTE|2.221094749172707...|
+------------+--------------------+

A região SUL apresentou o maior volume de crédito rural. Em seguida aparecem as regiões CENTRO-OESTE e SUDESTE.


In [8]:
# PERGUNTA 4 - QUAIS PROGRAMAS DE CRÉDITO CONCETRAM MAIS RECURSOS?

programa_sql = spark.sql("""
SELECT
    cdPrograma,
    SUM(VL_TOTAL) AS VolumeTotal
FROM credito_rural
GROUP BY cdPrograma
ORDER BY VolumeTotal DESC
""")

programa_sql.show()

+----------+--------------------+
|cdPrograma|         VolumeTotal|
+----------+--------------------+
|      0999|2.068981781479349E12|
|      0001|  5.0917362320888E11|
|      0050|4.377740646042690...|
|      0154|7.538598187348996E10|
|      0070|5.337181593550001...|
|      0163|3.354506752010002E10|
|      0156|2.470847920280999E10|
|      0162|   2.216134919425E10|
|      0157|2.118552493566002E10|
|      0222|1.838955945679997...|
|      0153|1.436442783551000...|
|      0151|1.390944226953999...|
|      0155|1.220438857855000...|
|      0152|1.035144302087000...|
|      0201|     2.02881750141E9|
|      0160|1.8408598239600003E9|
|      0161| 9.621348783700001E8|
|      0164|      2.4396089853E8|
|      0180|1.0039730760000002E8|
|      0165|       2.930446498E7|
+----------+--------------------+
only showing top 20 rows


In [9]:
# PERGUNTA 5 - QUAIS FONTES DE RECURSOS POSSUEM MAIOR PARTICIPAÇÃO?

fonte_sql = spark.sql("""
SELECT
    cdFonteRecurso,
    SUM(VL_TOTAL) AS VolumeTotal
FROM credito_rural
GROUP BY cdFonteRecurso
ORDER BY VolumeTotal DESC
""")

fonte_sql.show()

+--------------+--------------------+
|cdFonteRecurso|         VolumeTotal|
+--------------+--------------------+
|          0201|8.351198542469005E11|
|          0430|6.110426978206306E11|
|          0300|5.186656538245304...|
|          0303|3.042223471176200...|
|          0505|2.388088111804497...|
|          0402|1.649399511136298E11|
|          0502|1.428980319755395E11|
|          0403|9.796338698467973E10|
|          0503|7.851498591780014E10|
|          0501|   7.058289395177E10|
|          0431|6.911021494628006E10|
|          0800|5.334805706264001...|
|          0506|3.926388977824993E10|
|          0440|3.671485001725002E10|
|          0850|1.248339661143000...|
|          0450|1.086954592634999E10|
|          0222| 7.148015639329995E9|
|          0304| 6.062765294860002E9|
|          0301| 5.202112985680001E9|
|          0226|4.0172155250799975E9|
+--------------+--------------------+
only showing top 20 rows


In [10]:
# PERGUNTA 6 - QUAL MODALIDADE RECEBE MAIS RECURSOS?

modalidades = (
    df_spark
    .agg(
        sum("VlCusteio").alias("CUSTEIO"),
        sum("VlInvestimento").alias("INVESTIMENTO"),
        sum("VlComercializacao").alias("COMERCIALIZACAO"),
        sum("VlIndustrializacao").alias("INDUSTRIALIZACAO")
    )
)

dados = modalidades.collect()[0]

ranking = {
    "Custeio": dados["CUSTEIO"],
    "Investimento": dados["INVESTIMENTO"],
    "Comercialização": dados["COMERCIALIZACAO"],
    "Industrialização": dados["INDUSTRIALIZACAO"]
}

# Ordenar do maior para o menor
ranking_ordenado = sorted(
    ranking.items(),
    key=lambda x: x[1],
    reverse=True
)

# Função para formatar bilhões
def formatar_valor(valor):
    return f"R$ {valor/1_000_000_000:.1f} bilhões".replace(".", ",")

texto = (
    f"O ranking das modalidades de crédito rural é liderado por "
    f"{ranking_ordenado[0][0]} ({formatar_valor(ranking_ordenado[0][1])}), "
    f"seguido por {ranking_ordenado[1][0]} "
    f"({formatar_valor(ranking_ordenado[1][1])}), "
    f"{ranking_ordenado[2][0]} "
    f"({formatar_valor(ranking_ordenado[2][1])}) e "
    f"{ranking_ordenado[3][0]} "
    f"({formatar_valor(ranking_ordenado[3][1])})."
)

print(texto)

O ranking das modalidades de crédito rural é liderado por Custeio (R$ 1831,1 bilhões), seguido por Investimento (R$ 897,4 bilhões), Comercialização (R$ 418,6 bilhões) e Industrialização (R$ 173,6 bilhões).


In [11]:
# PERGUNTA 7: Evolução Mensal
evolucao_mensal = (
    df_spark
    .groupBy("MesEmissao")
    .agg(sum("VL_TOTAL").alias("VolumeTotal"))
    .orderBy("MesEmissao")
)

evolucao_mensal.show()

dados = evolucao_mensal.collect()

# Ordenar do maior para o menor volume
dados_ordenados = sorted(
    dados,
    key=lambda x: x["VolumeTotal"],
    reverse=True
)

primeiro = dados_ordenados[0]
segundo = dados_ordenados[1]

# Total de crédito de todos os meses
total_geral = builtins.sum(x["VolumeTotal"] for x in dados)

# Participação percentual
perc_primeiro = (primeiro["VolumeTotal"] / total_geral) * 100
perc_segundo = (segundo["VolumeTotal"] / total_geral) * 100

texto = (
    f"Os meses com maior volume de crédito rural foram "
    f"{primeiro['MesEmissao']} ({perc_primeiro:.2f}% do total analisado) "
    f"e {segundo['MesEmissao']} ({perc_segundo:.2f}% do total analisado)."
)

print(texto)

+----------+--------------------+
|MesEmissao|         VolumeTotal|
+----------+--------------------+
|        01|1.557772309219896...|
|        02|1.747251373490999E11|
|        03|2.431983200050195E11|
|        04|2.377509729527002...|
|        05|2.768069223711902E11|
|        06|3.439640251538705E11|
|        07|2.999776145999596E11|
|        08|4.419257307858908E11|
|        09|3.489959132052295...|
|        10|2.853609723466208E11|
|        11|2.451422299707501...|
|        12|2.670898408602795...|
+----------+--------------------+

Os meses com maior volume de crédito rural foram 08 (13.31% do total analisado) e 09 (10.51% do total analisado).


In [12]:
# Análise 6: Programas de Crédito

programa = (
    df_spark
    .groupBy("cdPrograma")
    .agg(sum("VL_TOTAL").alias("VolumeTotal"))
    .orderBy("VolumeTotal", ascending=False)
)

programa.show()

+----------+--------------------+
|cdPrograma|         VolumeTotal|
+----------+--------------------+
|      0999|2.068981781479349E12|
|      0001|  5.0917362320888E11|
|      0050|4.377740646042690...|
|      0154|7.538598187348996E10|
|      0070|5.337181593550001...|
|      0163|3.354506752010002E10|
|      0156|2.470847920280999E10|
|      0162|   2.216134919425E10|
|      0157|2.118552493566002E10|
|      0222|1.838955945679997...|
|      0153|1.436442783551000...|
|      0151|1.390944226953999...|
|      0155|1.220438857855000...|
|      0152|1.035144302087000...|
|      0201|     2.02881750141E9|
|      0160|1.8408598239600003E9|
|      0161| 9.621348783700001E8|
|      0164|      2.4396089853E8|
|      0180|1.0039730760000002E8|
|      0165|       2.930446498E7|
+----------+--------------------+
only showing top 20 rows


In [13]:
# ETAPA 4 - MODELAGEM PREDITIVA
from pyspark.sql.functions import when, col
from pyspark.ml.feature import StringIndexer, VectorAssembler

# CRIA A VARIÁVEL ALVO
df_ml = df_spark.withColumn(
    "FAIXA_CREDITO",
    when(col("VL_TOTAL") < 5000000000, 0)
    .when(col("VL_TOTAL") < 15000000000, 1)
    .otherwise(2)
)

# TRANSFORMA VARIÁVEIS CATEGÓRICAS EM NUMÉRICAS
indexador_uf = StringIndexer(
    inputCol="nomeUF",
    outputCol="UF_INDEX"
)
indexador_regiao = StringIndexer(
    inputCol="nomeRegiao",
    outputCol="REGIAO_INDEX"
)

df_ml = indexador_uf.fit(df_ml).transform(df_ml)
df_ml = indexador_regiao.fit(df_ml).transform(df_ml)

# Converte AnoEmissao e MesEmissao para tipo numérico
df_ml = df_ml.withColumn("AnoEmissao", col("AnoEmissao").cast("integer"))
df_ml = df_ml.withColumn("MesEmissao", col("MesEmissao").cast("integer"))

# CRIA O VETOR DE ATRIBUTOS

assembler_ml = VectorAssembler(
    inputCols=[
        "UF_INDEX",
        "REGIAO_INDEX",
        "AnoEmissao",
        "MesEmissao"
    ],
    outputCol="features"
)

df_ml = assembler_ml.transform(df_ml)

# DIVIDE TREINO E TESTE
treino, teste = df_ml.randomSplit(
    [0.8, 0.2],
    seed=42
)

In [14]:
# MODELO 1 - ÁRVORE DE DECISÃO
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    labelCol="FAIXA_CREDITO",
    featuresCol="features"
)

modelo_dt = dt.fit(treino)
pred_dt = modelo_dt.transform(teste)

In [15]:
# MODELO 2 - RANDOM FOREST (Ensemble)
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="FAIXA_CREDITO",
    featuresCol="features",
    numTrees=100
)

modelo_rf = rf.fit(treino)
pred_rf = modelo_rf.transform(teste)

In [16]:
# AVALIAÇÃO - MODELAGEM
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

avaliador_acc = MulticlassClassificationEvaluator(
    labelCol="FAIXA_CREDITO",
    predictionCol="prediction",
    metricName="accuracy"
)

avaliador_f1 = MulticlassClassificationEvaluator(
    labelCol="FAIXA_CREDITO",
    predictionCol="prediction",
    metricName="f1"
)

print("ACCURACY DT:", avaliador_acc.evaluate(pred_dt))
print("ACCURACY RF:", avaliador_acc.evaluate(pred_rf))

print("F1 DT:", avaliador_f1.evaluate(pred_dt))
print("F1 RF:", avaliador_f1.evaluate(pred_rf))

ACCURACY DT: 1.0
ACCURACY RF: 1.0
F1 DT: 1.0
F1 RF: 1.0


In [17]:
# ETAPA 5 - CLUSTERIZAÇÃO


df_cluster = (
    df_spark
    .groupBy("nomeUF", "nomeRegiao")
    .agg(
        sum("VlCusteio").alias("CUSTEIO"),
        sum("VlInvestimento").alias("INVESTIMENTO"),
        sum("VlComercializacao").alias("COMERCIALIZACAO"),
        sum("VlIndustrializacao").alias("INDUSTRIALIZACAO"),
        sum("VL_TOTAL").alias("TOTAL")
    )
)

df_cluster.show()

+------+------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|nomeUF|  nomeRegiao|             CUSTEIO|        INVESTIMENTO|     COMERCIALIZACAO|    INDUSTRIALIZACAO|               TOTAL|
+------+------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|    AC|       NORTE|     3.83169024892E9|3.6138590602299976E9|       1.133522431E7|      2.8696069779E8| 7.743845231249998E9|
|    MG|     SUDESTE|2.356193180127301...|1.048400656286400...|6.669085701990997E10|1.466686049239000...|4.218171011536693E11|
|    AL|    NORDESTE| 4.686217403749999E9|     4.94125763076E9|4.7573462605999994E8|2.8883264753999996E8|1.039204230810999...|
|    GO|CENTRO-OESTE|  1.7922139752788E11|7.511540050186003E10|3.931124301500998E10| 7.484071584800001E9|  3.0113211262955E11|
|    MS|CENTRO-OESTE|1.195064259647200...|4.628536313070996E10|1.583394136002999...|     2.72795068355E9|1.8435

In [18]:
# 5.2 MONTAR VETOR DE ATRIBUTOS
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col

if "features" in df_cluster.columns:
    df_cluster = df_cluster.drop("features")

assembler = VectorAssembler(
    inputCols=[
        "CUSTEIO",
        "INVESTIMENTO",
        "COMERCIALIZACAO",
        "INDUSTRIALIZACAO",
        "TOTAL"
    ],
    outputCol="features"
)

df_cluster = assembler.transform(df_cluster)


# 5.3 PADRONIZAR OS DADOS

from pyspark.ml.feature import StandardScaler

if "scaled_features" in df_cluster.columns:
    df_cluster = df_cluster.drop("scaled_features")

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features"
)

scholar_model = scaler.fit(df_cluster)
df_cluster = scholar_model.transform(df_cluster)


# 5.4 APLICAR O K-MEANS

from pyspark.ml.clustering import KMeans

kmeans = KMeans(
    k=3,
    seed=42,
    featuresCol="scaled_features"
)

modelo = kmeans.fit(df_cluster)

resultado = modelo.transform(df_cluster)

# Foi adotado K=3 por apresentar boa separação entre os grupos e um índice Silhouette satisfatório.

# AVALIAÇÃO DA CLUSTERIZAÇÃO - SILHOUETTE SCORE

from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    featuresCol="scaled_features",
    predictionCol="prediction"
)

silhouette = evaluator.evaluate(resultado)

print("Silhouette Score:", silhouette)

# 5.5 VISUALIZAR OS CLUSTERS

resultado.select(
    "nomeUF",
    "nomeRegiao",
    "prediction"
).show(50, False)

# 5.5 VISUALIZAR OS CLUSTERS
resultado.select(
    "nomeUF",
    "nomeRegiao",
    "prediction" # Alterado de "cluster" para "prediction"
).show(50, False)


Silhouette Score: 0.8407107523739676
+------+------------+----------+
|nomeUF|nomeRegiao  |prediction|
+------+------------+----------+
|AC    |NORTE       |0         |
|MG    |SUDESTE     |1         |
|AL    |NORDESTE    |0         |
|GO    |CENTRO-OESTE|1         |
|MS    |CENTRO-OESTE|0         |
|BA    |NORDESTE    |0         |
|RR    |NORTE       |0         |
|PA    |NORTE       |0         |
|AM    |NORTE       |0         |
|PE    |NORDESTE    |0         |
|MT    |CENTRO-OESTE|1         |
|PB    |NORDESTE    |0         |
|PI    |NORDESTE    |0         |
|MA    |NORDESTE    |0         |
|RJ    |SUDESTE     |0         |
|SC    |SUL         |0         |
|DF    |CENTRO-OESTE|0         |
|PR    |SUL         |2         |
|RS    |SUL         |1         |
|SE    |NORDESTE    |0         |
|CE    |NORDESTE    |0         |
|RN    |NORDESTE    |0         |
|RO    |NORTE       |0         |
|AP    |NORTE       |0         |
|ES    |SUDESTE     |0         |
|SP    |SUDESTE     |1         |
|TO   

In [19]:
# ETAPA 6 INTERPRETAÇÃO (KDD)

resultado.groupBy("prediction").count().show()

# Cluster 0: Estados com alto volume de crédito rural.
# Cluster 1: Estados com volume intermediário e perfil diversificado.
# Cluster 2: Estados com menor participação no crédito rural.

+----------+-----+
|prediction|count|
+----------+-----+
|         1|    5|
|         2|    1|
|         0|   21|
+----------+-----+

